# Model exploration - MS2

Before wiring anything into the FastAPI service or the XR app, run each model
here and look at its output. The goal is to answer **"does it work on this
machine, and what does it actually produce?"**

| Section | Model | Purpose in the project |
|---|---|---|
| 1 | YOLO11n | find objects in the frame |
| 2 | FastSAM-s | fast class-agnostic masks |
| 3 | MobileSAM | promptable mask under the reticle |
| 4 | Depth Anything V2 | depth -> lift masks to 3D |
| 5 | CLIP | zero-shot classify / label |
| 6 | Florence-2 (optional) | caption / "scanned" text |
| 7 | pipeline | unproject a detection to a 3D point cloud |

**First run downloads weights** (YOLO11n ~5 MB, FastSAM-s ~23 MB, MobileSAM
~40 MB, Depth-Anything-V2-Small ~100 MB, CLIP ~600 MB). CPU is fine for a first
look; OpenVINO export is discussed at the end.

## 0. Setup

In [ ]:
# Install once, from services/inference/notebooks:
#   pip install -r requirements.txt
# (or uncomment the line below in the notebook)
# %pip install -r requirements.txt

In [ ]:
import sys, time, urllib.request
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import torch

print("python ", sys.version.split()[0])
print("torch  ", torch.__version__)
print("cuda   ", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device ", DEVICE)

SAMPLES = Path("samples")
SAMPLES.mkdir(exist_ok=True)

In [ ]:
%matplotlib inline

## 1. Sample images

In [ ]:
SAMPLE_URLS = {
    # Ultralytics' classic test images
    "bus.jpg": "https://ultralytics.com/images/bus.jpg",
    "zidane.jpg": "https://ultralytics.com/images/zidane.jpg",
}


def fetch(name: str) -> Image.Image:
    path = SAMPLES / name
    if not path.exists():
        print("downloading", name)
        urllib.request.urlretrieve(SAMPLE_URLS[name], path)
    return Image.open(path).convert("RGB")


imgs = {name: fetch(name) for name in SAMPLE_URLS}

fig, axes = plt.subplots(1, len(imgs), figsize=(12, 4))
for ax, (name, im) in zip(np.atleast_1d(axes), imgs.items()):
    ax.imshow(im)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()

## 2. YOLO11n - object detection

The detector gives *what* and *where* (boxes + class + confidence). This runs on
every keyframe in the real pipeline, so its latency matters most.

In [ ]:
from ultralytics import YOLO

t0 = time.time()
det = YOLO("yolo11n.pt")  # auto-downloads on first use
print(f"loaded yolo11n in {time.time() - t0:.2f}s")

for name, im in imgs.items():
    t0 = time.time()
    res = det.predict(np.array(im), imgsz=640, verbose=False)[0]
    dt = (time.time() - t0) * 1000
    print(f"{name}: {len(res.boxes)} boxes in {dt:.0f} ms")
    for b in res.boxes[:10]:
        print(f"    {res.names[int(b.cls)]:<12} {float(b.conf):.2f}")

In [ ]:
res = det.predict(np.array(imgs["bus.jpg"]), imgsz=640, verbose=False)[0]
plt.figure(figsize=(9, 6))
plt.imshow(res.plot()[:, :, ::-1])  # plot() returns BGR
plt.axis("off")
plt.title(f"yolo11n - {len(res.boxes)} detections")

## 3. FastSAM-s - fast class-agnostic segmentation

Gives a mask per region. Much faster than SAM ViT-H, at some quality cost. This
is our default "segment the whole frame" model.

In [ ]:
from ultralytics import FastSAM

t0 = time.time()
fastsam = FastSAM("FastSAM-s.pt")
print(f"loaded FastSAM-s in {time.time() - t0:.2f}s")

t0 = time.time()
fs_res = fastsam(np.array(imgs["bus.jpg"]), imgsz=1024, conf=0.4, iou=0.9, verbose=False)[0]
print(f"inference {((time.time() - t0) * 1000):.0f} ms, {len(fs_res.boxes)} masks")

plt.figure(figsize=(9, 6))
plt.imshow(fs_res.plot()[:, :, ::-1])
plt.axis("off")
plt.title("FastSAM-s segments")

## 4. MobileSAM - promptable segmentation (the reticle case)

When the user aims the reticle at an object we know roughly *where* it is, so a
point prompt is cheaper and cleaner than segmenting everything. This is the
model to call for the "scan the object under the crosshair" flow.

> Note the point below is hard-coded for the demo; in the app it would be the
> screen centre.

In [ ]:
from ultralytics import SAM

sam = SAM("mobile_sam.pt")
im = imgs["bus.jpg"]
w, h = im.size
centre = [w // 2, h // 2]

t0 = time.time()
sam_res = sam(np.array(im), points=[centre], labels=[1], verbose=False)[0]
print(f"mobile_sam {((time.time() - t0) * 1000):.0f} ms, "
      f"{0 if sam_res.masks is None else len(sam_res.masks)} mask(s) for point {centre}")

plt.figure(figsize=(9, 6))
plt.imshow(sam_res.plot()[:, :, ::-1])
plt.scatter([centre[0]], [centre[1]], c="red", s=40, marker="+")
plt.axis("off")
plt.title("MobileSAM point-prompt")

## 5. Depth Anything V2 Small - monocular depth

No depth sensing on our test phone, so this is how we lift a 2D mask into 3D.
Output is **relative** depth (not metric) - fine for shaping an outline, but the
absolute scale has to be fixed some other way (e.g. known object size or planes).

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"
depth_proc = AutoImageProcessor.from_pretrained(DEPTH_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_ID).to(DEVICE).eval()


@torch.no_grad()
def depth_map(im: Image.Image) -> np.ndarray:
    inputs = depth_proc(images=im, return_tensors="pt").to(DEVICE)
    out = depth_model(**inputs).predicted_depth  # (1, h, w)
    out = torch.nn.functional.interpolate(
        out.unsqueeze(1), size=(im.height, im.width), mode="bicubic", align_corners=False
    )
    return out.squeeze().cpu().numpy()


d = depth_map(imgs["bus.jpg"])
print("depth", d.shape, "min", float(d.min()), "max", float(d.max()))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(imgs["bus.jpg"])
plt.axis("off")
plt.title("rgb")
plt.subplot(1, 2, 2)
plt.imshow(d, cmap="magma")
plt.axis("off")
plt.title("Depth Anything V2 (near = bright)")
plt.tight_layout()

## 6. CLIP ViT-B/32 - zero-shot classification

Labels the cropped object without any training. We can edit `PROMPTS` freely to
change the taxonomy, and later swap in a richer captioner.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)

PROMPTS = [
    "a photo of a person",
    "a photo of a bus",
    "a photo of a cup",
    "a photo of a chair",
    "a photo of a laptop",
    "a photo of a wall",
    "a photo of a phone",
]


@torch.no_grad()
def classify(im: Image.Image):
    inputs = clip_proc(text=PROMPTS, images=im, return_tensors="pt", padding=True).to(DEVICE)
    probs = clip_model(**inputs).logits_per_image.softmax(dim=-1)[0]
    pairs = sorted(zip(PROMPTS, probs.tolist()), key=lambda p: -p[1])
    return pairs


for im, label in [(imgs["bus.jpg"], "bus.jpg"), (imgs["zidane.jpg"], "zidane.jpg")]:
    top = classify(im)[:3]
    print(label, "->", ", ".join(f"{p.split()[-1]} {s:.2f}" for p, s in top))

## 7. Florence-2 (optional) - caption for the scan card

This is the "scanned" text. It is heavier than the rest; uncomment to try. If it
is too slow on CPU, that is a signal to run it server-side only, or drop it for
a template like `"<label>, confidence <x>"`.

In [ ]:
# from transformers import AutoProcessor, AutoModelForCausalLM
# FLORENCE_ID = "microsoft/Florence-2-base-ft"
# fl_proc = AutoProcessor.from_pretrained(FLORENCE_ID, trust_remote_code=True)
# fl_model = AutoModelForCausalLM.from_pretrained(FLORENCE_ID, trust_remote_code=True).to(DEVICE).eval()

# def caption(im):
#     task = "<MORE_DETAILED_CAPTION>"
#     inputs = fl_proc(text=task, images=im, return_tensors="pt").to(DEVICE)
#     with torch.no_grad():
#         ids = fl_model.generate(**inputs, max_new_tokens=64, num_beams=3)
#     return fl_proc.batch_decode(ids, skip_special_tokens=False)[0]

# print(caption(imgs["bus.jpg"]))

## 8. Putting it together: unproject a detection to 3D

The core trick of the whole project: take a detection box, read the depth over
that region, and back-project the pixels through a pinhole camera model to get a
3D point cluster. Fit an oriented box to that and you have the **scan border**
that can be anchored in the world.

Caveats to keep in mind:
- Depth is relative, so the cloud will have an arbitrary scale here. The app
  fixes scale using AR planes / a known reference.
- Intrinsics are guessed from an assumed FOV; the real app gets them from WebXR.

In [ ]:
def intrinsics_from_fov(width: int, height: int, fov_deg: float = 60.0):
    fx = 0.5 * width / np.tan(np.radians(fov_deg) / 2.0)
    fy = fx
    return fx, fy, width / 2.0, height / 2.0


def unproject_box(box, depth, stride=4):
    h, w = depth.shape
    fx, fy, cx, cy = intrinsics_from_fov(w, h)
    x0, y0, x1, y1 = (int(v) for v in box)
    ys, xs = np.mgrid[y0:y1:stride, x0:x1:stride]
    z = depth[ys, xs]
    x = (xs - cx) * z / fx
    y = (ys - cy) * z / fy
    pts = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    return pts, z.reshape(-1)


# largest detection on the bus image
res = det.predict(np.array(imgs["bus.jpg"]), imgsz=640, verbose=False)[0]
boxes = sorted(res.boxes, key=lambda b: -(float(b.xyxy[0][2] - b.xyxy[0][0]) * float(b.xyxy[0][3] - b.xyxy[0][1])))
target = boxes[0]
label = res.names[int(target.cls)]
box = target.xyxy[0].tolist()
pts, z = unproject_box(box, d)

# drop the far background using a depth quantile band
lo, hi = np.percentile(z, 5), np.percentile(z, 80)
keep = (z >= lo) & (z <= hi)
pts = pts[keep]
print(f"target '{label}' box={[round(v) for v in box]} -> {len(pts)} points")

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1.5, c=-pts[:, 2], cmap="viridis")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z (relative)")
ax.set_title(f"3D point cloud for '{label}'")
ax.invert_yaxis()
plt.tight_layout()

## 9. Where OpenVINO comes in

The service will export the models to OpenVINO and run them on the Intel iGPU.
Export is a one-liner for Ultralytics models and an `optimum-intel` call for the
HuggingFace ones. Try it here and compare CPU vs GPU latency:

```python
# YOLO
# det.export(format="openvino", half=True)   # -> yolo11n_openvino_model/
# fastsam.export(format="openvino")
#
# then load with device="intel:gpu":
# from ultralytics import YOLO
# ov = YOLO("yolo11n_openvino_model/", task="detect")
# ov.predict(..., device="intel:gpu")

# HuggingFace depth/CLIP
# from optimum.intel import OVModelForDepthEstimation, OVModelForZeroShotImageClassification
```

## 10. Next steps

- Note which models were acceptably fast and where quality is good enough.
- Decide the final taxonomy for `PROMPTS`.
- Then we turn this notebook into the FastAPI service (`services/inference/app`)
  with `/scan`, `/depth`, `/classify` endpoints.